# Telco Customer Churn Prediction

## What is this project about?
This project uses Machine Learning to predict if a customer will leave a telecom company (Churn). 

## Why is it important?
It is cheaper to keep an old customer than to find a new one. If we know who wants to leave, the company can give them a discount and save them.

## Dataset and Metric
* Data: Telco Customer Churn (Kaggle).
* Main metric: **Recall**. We care more about finding people who will leave, even if we make some false alarms.

## 0. Setup and Imports

In [1]:
import pandas as pd
import numpy as np

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier

from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.metrics import classification_report

data = pd.read_csv("/kaggle/input/datasets/blastchar/telco-customer-churn/WA_Fn-UseC_-Telco-Customer-Churn.csv")

## 1. Exploratory Data Analysis 

First, we load the data and look at it. 
We need to check two things:
* Are the classes balanced?
* Do we have missing values (`NaN`)?

In [2]:
data.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [3]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 non-null   object 
 15  Contract          7043 non-null   object 
 16  PaperlessBilling  7043 non-null   object 


In [4]:
data['Churn'].value_counts(normalize=True)

Churn
No     0.73463
Yes    0.26537
Name: proportion, dtype: float64

In [5]:
data.isna().sum()

customerID          0
gender              0
SeniorCitizen       0
Partner             0
Dependents          0
tenure              0
PhoneService        0
MultipleLines       0
InternetService     0
OnlineSecurity      0
OnlineBackup        0
DeviceProtection    0
TechSupport         0
StreamingTV         0
StreamingMovies     0
Contract            0
PaperlessBilling    0
PaymentMethod       0
MonthlyCharges      0
TotalCharges        0
Churn               0
dtype: int64

### Wait, `isna()` shows 0 missing values!

Let's look at `data.info()`. We can see that `TotalCharges` has the type **`object`** (text), but it should be a number. 

Why is it text? Because it contains **spaces** (`" "`) instead of real numbers. Pandas thinks it's just text, so `isna()` doesn't see them as missing values. 

We need to convert this column to numbers first.

## 2. Data Preprocessing

Here we prepare the data for the model:
1. We fix `TotalCharges` (it has spaces instead of numbers).
2. We split the data into Train and Test. We use `stratify=y` to keep the same percentage of churners in both sets.
3. We build a `Pipeline` to scale numbers and encode text automatically.

In [6]:
X = data.drop(['customerID', 'Churn'], axis=1)
y = data['Churn'].map({'Yes': 1, 'No': 0})

X['TotalCharges'] = pd.to_numeric(X['TotalCharges'], errors='coerce')

# Check again to verify
print(X.isna().sum())

gender               0
SeniorCitizen        0
Partner              0
Dependents           0
tenure               0
PhoneService         0
MultipleLines        0
InternetService      0
OnlineSecurity       0
OnlineBackup         0
DeviceProtection     0
TechSupport          0
StreamingTV          0
StreamingMovies      0
Contract             0
PaperlessBilling     0
PaymentMethod        0
MonthlyCharges       0
TotalCharges        11
dtype: int64


### Fixed!
Now we see the real 11 missing values. Our `SimpleImputer` inside the Pipeline will handle them later.

In [7]:
num_cols = X.select_dtypes(include=['number']).columns.tolist()
cat_cols = X.select_dtypes(include=['object']).columns.tolist()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y 
)

In [8]:
pipeline_numeric = Pipeline([
    ('Imputer', SimpleImputer(strategy='median')),
    ('Scaler', StandardScaler())
])

pipeline_cat = Pipeline([
    ('Imputer', SimpleImputer(strategy='most_frequent')),
    ('Encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer([
    ('Num', pipeline_numeric, num_cols),
    ('Cat', pipeline_cat, cat_cols)
])

## 3. Baseline Models Comparison

We test 5 different models using Cross-Validation (`cv=5`).
Our main metric is **Recall**. We want to find as many churners as possible. 
We don't care if we make some false alarms (give a discount to a loyal client).

In [9]:
pipeline_rf = Pipeline([
    ('Prep', preprocessor),
    ('Model', RandomForestClassifier(max_depth=5, class_weight='balanced', random_state=42))
])

pipeline_lr = Pipeline([
    ('Prep', preprocessor),
    ('Model', LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42))
])

pipeline_gb = Pipeline([
    ('Prep', preprocessor),
    ('Model', GradientBoostingClassifier(random_state=42))
])

pipeline_svc = Pipeline([
    ('Prep', preprocessor),
    ('Model', SVC(class_weight='balanced', random_state=42))
])

pipeline_knn = Pipeline([
    ('Prep', preprocessor),
    ('Model', KNeighborsClassifier(n_neighbors=5))
])

models = {
    'RandomForest': pipeline_rf,
    'LogisticRegression': pipeline_lr,
    'GradientBoosting': pipeline_gb,
    'SVM': pipeline_svc,
    'KNN': pipeline_knn
}

In [10]:
# Run 5-fold cross-validation
for name, model in models.items():
    cv_scores = cross_val_score(model, X_train, y_train, cv=5, scoring='recall', n_jobs=-1)
    print(f"{name}: Recall = {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")

RandomForest: Recall = 0.7960 ± 0.0178
LogisticRegression: Recall = 0.8047 ± 0.0207
GradientBoosting: Recall = 0.5204 ± 0.0314
SVM: Recall = 0.7793 ± 0.0118
KNN: Recall = 0.5331 ± 0.0297


## 4. Hyperparameter Tuning and Final Evaluation

Logistic Regression showed the best baseline score. 
Now we use `GridSearchCV` to find the best parameter `C` for it. 
Finally, we check the model on the 20% test data we saved at the beginning.

In [11]:
param_grid_lr = {
    'Model__C': [0.01, 0.1, 1, 10, 100],
    'Model__solver': ['liblinear', 'lbfgs']
}

grid_search_lr = GridSearchCV(
    estimator=pipeline_lr,
    param_grid=param_grid_lr,
    cv=5,
    scoring='recall',
    n_jobs=-1,
    verbose=1
)

grid_search_lr.fit(X_train, y_train);

Fitting 5 folds for each of 10 candidates, totalling 50 fits


In [12]:
print(f"\nBest parameters: {grid_search_lr.best_params_}")
print(f"Best CV Recall: {grid_search_lr.best_score_:.4f}")

# Final check on test data
y_pred_final = grid_search_lr.predict(X_test)

print(classification_report(y_test, y_pred_final))


Best parameters: {'Model__C': 1, 'Model__solver': 'liblinear'}
Best CV Recall: 0.8047
              precision    recall  f1-score   support

           0       0.90      0.72      0.80      1035
           1       0.50      0.78      0.61       374

    accuracy                           0.74      1409
   macro avg       0.70      0.75      0.71      1409
weighted avg       0.80      0.74      0.75      1409



## 5. Conclusion

We built a Machine Learning model to predict customer churn.
* The best model is **Logistic Regression** with `C=1`.
* On the test data, our **Recall is 0.78**. This means we can find 78% of customers who want to leave.
* The model makes some false alarms (Precision is 0.50), but it is better to give a discount to a loyal client than to lose a real customer.

### Next steps:
* Try gradient boosting models (XGBoost / LightGBM) to improve precision.
* Add more features (like customer call center history).